# C3 - Day 3: Logistic Regression Baseline
**AnchorAI | Component 3: Cognitive Checkpointing**

Goal: train the simplest sensible classifier for **resumption difficulty (Low / Medium / High)** and measure it honestly. A *baseline* is the model every later model (Decision Tree, Random Forest) has to beat.

We report three things:
1. **Cross-validation on training participants** (5 participant-wise folds) - how stable is the result?
2. **Final test on unseen participants** - used once.
3. **Feature-group ablation** - which kind of information actually helps?

> Run on **CPU runtime**. Needs `samples.csv` and `meta.json` from Day 2 in `MyDrive/C3/data/processed/`.

## Step 0 - Mount Drive, load the Day 2 data

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

DATA_DIR   = os.path.join(BASE, 'data', 'raw', 'hirsch')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

S = pd.read_csv(os.path.join(PROC_DIR, 'samples.csv'))
with open(os.path.join(PROC_DIR, 'meta.json')) as f:
    meta = json.load(f)
LABEL_NAMES = meta['label_names']                  # 0 = Low, 1 = Medium, 2 = High

train = S[S['split'] == 'train'].reset_index(drop=True)
test  = S[S['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train):,} trials, {train.pp_id.nunique()} participants | Test: {len(test):,} trials, {test.pp_id.nunique()} participants')
print('Label names:', dict(enumerate(LABEL_NAMES)))

## Step 1 - Feature groups
We group the features by the *kind* of information they carry. Step 5 adds the groups one at a time to see what each contributes.

| Group | Features |
|-------|----------|
| design | `sequence`, `block`, `trial`, `experiment` |
| interruption | `interruption_rt`, `interruption_error` |
| pace | `rt_subtask1`, `rt_subtask2`, `pre_errors` |
| person | `pp_speed` |

In [ ]:
GROUPS = {
    'design':       ['sequence', 'block', 'trial', 'experiment'],
    'interruption': ['interruption_rt', 'interruption_error'],
    'pace':         ['rt_subtask1', 'rt_subtask2', 'pre_errors'],
    'person':       ['pp_speed'],
}
ALL_FEATURES = [c for g in GROUPS.values() for c in g]
print(ALL_FEATURES)

## Step 2 - The models
Logistic Regression learns one weight per feature. Two preparation steps are needed:
* **Scaling** the numeric features (mean 0, spread 1): otherwise a feature measured in thousands of ms (like `interruption_rt`) dominates one measured in 0/1.
* **One-hot encoding** `experiment` (Exp1/Exp2/Exp3 are names, not quantities).

The preparation is learned on the training part only, then applied to the validation/test part (a `Pipeline` does this automatically).

The **Dummy** model ignores the features and always predicts the most common class. It shows what "no learning" scores.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

def make_prep(features):
    cat = [c for c in features if c == 'experiment']
    num = [c for c in features if c != 'experiment']
    parts = [('num', StandardScaler(), num)]
    if cat:
        parts.append(('cat', OneHotEncoder(handle_unknown='ignore'), cat))
    return ColumnTransformer(parts)

def make_lr(features):
    return Pipeline([('prep', make_prep(features)), ('clf', LogisticRegression(max_iter=2000, random_state=42))])

def make_dummy(features):
    return Pipeline([('prep', make_prep(features)), ('clf', DummyClassifier(strategy='most_frequent'))])

## Step 3 - Cross-validation on training participants
The training participants were divided into 5 folds in Day 2. For each fold: train on the other 4, validate on that fold. Because a participant's trials never cross folds, every validation score is measured on **unseen people**.

**Macro-F1** is our main metric: it averages the F1 score of Low, Medium and High with equal weight. With 3 balanced classes, guessing gives about 33% accuracy.

In [ ]:
def cv_scores(make_model, features):
    rows = []
    for k in sorted(train['fold'].unique()):
        tr, va = train[train['fold'] != k], train[train['fold'] == k]
        m = make_model(features).fit(tr[features], tr['label'])
        p = m.predict(va[features])
        rows.append((accuracy_score(va['label'], p), f1_score(va['label'], p, average='macro')))
    return pd.DataFrame(rows, columns=['accuracy', 'macro_f1'])

cv = {'Dummy (most frequent)': cv_scores(make_dummy, ALL_FEATURES),
      'Logistic Regression':   cv_scores(make_lr, ALL_FEATURES)}
cv_table = pd.DataFrame({name: {'accuracy': f'{d.accuracy.mean():.3f} +/- {d.accuracy.std():.3f}',
                                'macro_f1': f'{d.macro_f1.mean():.3f} +/- {d.macro_f1.std():.3f}'} for name, d in cv.items()}).T
print('5-fold participant-wise cross-validation (mean +/- std over folds):')
print(cv_table)

## Step 4 - Final test on unseen participants
Now we train on **all** training participants and test **once** on the held-out test participants.

Metrics:
* **Precision (macro)**: when the model predicts a class, how often is it right (averaged over classes).
* **Recall (macro)**: of the real cases of a class, how many are found (averaged).
* **Severe error rate**: how often a *Low* is predicted as *High* or the reverse. The classes are ordered, so missing by two steps is worse than missing by one.

In [ ]:
def evaluate(make_model, name):
    m = make_model(ALL_FEATURES).fit(train[ALL_FEATURES], train['label'])
    p = m.predict(test[ALL_FEATURES])
    pr, rc, f1, _ = precision_recall_fscore_support(test['label'], p, average='macro', zero_division=0)
    return m, p, {'model': name, 'accuracy': accuracy_score(test['label'], p), 'precision_macro': pr,
                  'recall_macro': rc, 'f1_macro': f1,
                  'severe_error_rate': float((np.abs(test['label'].values - p) == 2).mean())}

lr_model, pred_lr, res_lr = evaluate(make_lr, 'Logistic Regression')
_, _, res_dummy = evaluate(make_dummy, 'Dummy (most frequent)')
results = pd.DataFrame([res_dummy, res_lr]).round(3)
print(results.to_string(index=False))
print()
print(classification_report(test['label'], pred_lr, target_names=LABEL_NAMES, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(test['label'], pred_lr, display_labels=LABEL_NAMES, cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Logistic Regression - test participants')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lr_confusion_matrix.png'), dpi=150)
plt.show()

## Step 5 - Feature-group ablation: what information actually helps?
We add the feature groups one at a time and re-run the cross-validation. If the score rises when a group is added, that kind of information is useful for predicting resumption difficulty.

In [ ]:
steps = [('design only', ['design']),
         ('+ interruption', ['design', 'interruption']),
         ('+ pace before interruption', ['design', 'interruption', 'pace']),
         ('+ person speed (all)', ['design', 'interruption', 'pace', 'person'])]
abl = []
for name, groups in steps:
    feats = [c for g in groups for c in GROUPS[g]]
    d = cv_scores(make_lr, feats)
    abl.append({'features': name, 'macro_f1_mean': d.macro_f1.mean(), 'macro_f1_std': d.macro_f1.std(), 'accuracy_mean': d.accuracy.mean()})
abl = pd.DataFrame(abl)
print(abl.round(3).to_string(index=False))

plt.figure(figsize=(7, 3.8))
plt.barh(abl['features'][::-1], abl['macro_f1_mean'][::-1], xerr=abl['macro_f1_std'][::-1], color='steelblue')
plt.axvline(cv['Dummy (most frequent)'].macro_f1.mean(), color='gray', ls='--', label='Dummy')
plt.xlabel('Macro-F1 (5-fold CV)'); plt.title('Logistic Regression: adding feature groups'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lr_ablation.png'), dpi=150)
plt.show()

## Step 6 - What did the model learn? (coefficients)
Features were scaled, so the weights are comparable. For the **High** class, a positive weight means "when this feature is larger, a High resumption cost becomes more likely".

In [ ]:
names = lr_model.named_steps['prep'].get_feature_names_out()
coef = lr_model.named_steps['clf'].coef_
high = pd.Series(coef[2], index=[n.split('__')[1] for n in names]).sort_values()
print('Weights for the High class (standardised):')
print(high.round(3).to_string())

plt.figure(figsize=(7, 4.5))
plt.barh(high.index, high.values, color=['#d9534f' if v > 0 else '#5b9bd5' for v in high.values])
plt.axvline(0, color='black', lw=1)
plt.title('Logistic Regression - weights for class High')
plt.xlabel('Weight (positive = more likely High cost)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lr_coefficients_high.png'), dpi=150)
plt.show()

## Step 7 - Save the model and the results
The results file is reused on Day 4 for the Logistic Regression vs Decision Tree vs Random Forest comparison.

In [ ]:
joblib.dump(lr_model, os.path.join(MODELS_DIR, 'lr_baseline.pkl'))
results.to_csv(os.path.join(OUTPUT_DIR, 'results_day3_lr.csv'), index=False)
abl.to_csv(os.path.join(OUTPUT_DIR, 'ablation_day3_lr.csv'), index=False)
print('Saved model, test results and ablation table.')

## Summary - Day 3
Read the numbers printed above:
* The Dummy model shows the score of "no learning" (accuracy near 33%, macro-F1 near 0.16).
* If Logistic Regression is clearly above it in **both** cross-validation and the final test, the features carry real signal about resumption difficulty.
* The ablation shows which kinds of information help.
* Compare the per-class recall in the classification report and the confusion matrix: the class with the lowest recall is the one the model finds hardest to recognise.

**Next: Day 4 - Decision Tree and Random Forest on the same folds, then a comparison table.**